In [ ]:
import pandas as pd
import numpy as np
import requests
from tqdm.notebook import tqdm
import os
import pickle

os.chdir("/home/cc/phd/KGEmbeddings")

from codes.query_solver import GeometricSolver
from codes.triplets import TripletsEngine

# PATH = "/home/marco_dossena/PHD/KGEmbeddings/"
PATH = "/home/cc/phd/KGEmbeddings/"
EMBEDDING_DIM = 512
DATA = "FB15k"
MODEL_NAME = "TransE"
# MODEL_PATH = "/home/cc/phd/KGEmbeddings/models/TransE_FB15k_0/"
# MODEL_PATH = "/home/cc/phd/KGEmbeddings/models/RotatE_FB15k_0/"
MODEL_PATH = f"{PATH}models/{MODEL_NAME}_{DATA}_0"
# DICTS_DIR = "/home/cc/phd/KGEmbeddings/data/FB15k/"
DICTS_DIR = f"{PATH}data/{DATA}"

e_map = pd.read_json("/home/cc/phd/KGEmbeddings/data/umls/entity_map.json", typ='series').to_dict()
r_map = pd.read_json("/home/cc/phd/KGEmbeddings/data/umls/rel_map.json", typ='series').to_dict()

inv_e_map = {v: k for k, v in e_map.items()}
inv_r_map = {v: k for k, v in r_map.items()}

# umls = pd.read_csv("/home/cc/phd/KGEmbeddings/data/umls/train.csv", low_memory=False)
# umls_r5 = umls[umls['relation_id'] == 5]

# number_of_proj = 2
# shared_tails = umls_r5.groupby('tail_id')['head_id'].nunique()
# shared_tails = shared_tails[shared_tails > (number_of_proj-1)]  # tails with more than number_of_proj heads

# shared_tails = shared_tails.sample(frac=1, random_state=77)  # shufflle series

kg = TripletsEngine(os.path.join(DICTS_DIR), ext="txt" if DATA == "FB15k" else "csv", from_splits=True)
qs = GeometricSolver(MODEL_PATH, MODEL_NAME.lower(), EMBEDDING_DIM, h2t=kg.h2t, t2h=kg.t2h, k_neighbors=50, k_results=25, device='cuda')

In [ ]:
# FB15k version

ent_map_fb = pd.read_csv("/home/cc/phd/KGEmbeddings/data/FB15k/entities.txt", header=None, sep="\t", low_memory=False)
ent_rel_fb = pd.read_csv("/home/cc/phd/KGEmbeddings/data/FB15k/relations.txt", header=None, sep="\t", low_memory=False)

keys = ent_map_fb[1]
values = ent_map_fb[0]
ent_map = dict(zip(keys, values))

keys = ent_rel_fb[1]
values = ent_rel_fb[0]
rel_map = dict(zip(keys, values))

fb = pd.read_csv("/home/cc/phd/KGEmbeddings/data/FB15k/train.txt", header=None, sep="\t", low_memory=False)
fb.columns = ['head_id', 'relation_id', 'tail_id']

fb['head_id'] = fb['head_id'].map(ent_map)
fb['relation_id'] = fb['relation_id'].map(rel_map)
fb['tail_id'] = fb['tail_id'].map(ent_map)

umls_r5 = fb
umls = fb

number_of_proj = 2
shared_tails = umls_r5.groupby('tail_id')['head_id'].nunique()
shared_tails = shared_tails[shared_tails > (number_of_proj-1)]  # tails with more than number_of_proj heads

shared_tails = shared_tails.sample(frac=1, random_state=77)  # shufflle series

In [ ]:
def api_call(cui):
    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}?apiKey=f72ff16d-f1da-40a6-adbc-9f42ff7c9fe7"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data.get('result', {}).get('name', 'N/A')
    else:
        return 'N/A'

def print_query(query, res):
    h1, r1 = query[0][0]
    h2, r2 = query[0][1]
    last_rel = query[1]
    print(f"({api_call(inv_e_map[h1])} - [{inv_r_map[r1]}]-> ?)")
    print(f"AND ({api_call(inv_e_map[h2])} - [{inv_r_map[r2]}]-> ?)")
    print(f"AND (?* - [{inv_r_map[last_rel]}]-> ?')")
    print(f"Results: {[api_call(inv_e_map[r]) for r in res]}")

def print_metrics(metrics, name, interval):
    print(f"{name} over {interval} queries:")
    for k in interval:
        print(f"{name}@{k}:{np.mean(metrics[f'{name.lower()}@{k}']):.4f}")
    print("-----------------------")

def recall_at_k(pred, true, k):
    if len(true) == 0:
        return 1.0
    
    if k > 0:
        pred_k = pred[:max(k, len(true))]
    else:
        pred_k = pred

    hits = sum([1 for p in pred_k if p in true])
    return hits / len(true)

def map_at_k(pred, true, k):
    if len(true) == 0:
        return 1.0
    
    if k > 0:
        pred_k = pred[:max(k, len(true))]
    else:
        pred_k = pred

    hits = sum([1 for p in pred_k if p in true])
    return hits / max(k, len(true))

def custom_hits_at_k(pred, trues, k):
    if len(trues) == 0:
        print("No true answers available.")
        return 1.0

    if len(pred) == 0:
        print("No predictions made.")
        return 0.0
    
    hits = []

    if len(trues) == 1:
        hits.append(1.0 if trues[0] in pred[:k] else 0.0)
    else:
        for true in trues:
            pred_set = np.setdiff1d(pred, trues[trues != true])
            hits.append(1.0 if true in pred_set[:k] else 0.0)

    return np.mean(hits)

def custom_mrr(pred, trues):
    if len(trues) == 0:
        return 1.0
    
    rr = []
    if len(trues) == 1:
        if trues[0] in pred:
            rank = np.where(pred == trues[0])[0][0] + 1
            rr.append(1.0 / rank)
        else:
            rr.append(0.0)
    else:
        for true in trues:
            pred_set = np.setdiff1d(pred, trues[trues != true])
            if true in pred_set:
                rank = np.where(pred_set == true)[0][0] + 1
                rr.append(1.0 / rank)
            else:
                rr.append(0.0)

    return np.mean(rr)

In [ ]:
queries = [] 
results = []  

if shared_tails.empty:
    print("No shared tails found with relation_id = 5")
else:
    for shared_tail_id in tqdm(shared_tails.index, desc="Processing queries"):
        # Get the heads pointing to this shared tail
        heads = umls_r5[umls_r5['tail_id'] == shared_tail_id]['head_id'].unique()[:number_of_proj]
        if len(heads) < number_of_proj:
            continue  # need at least number_of_proj heads

        relations = umls_r5[umls_r5['tail_id'] == shared_tail_id]['relation_id'].values

        # Save query structure (5 is the relation_id for "is_associated_with")
        query = [[(heads[i], relations[i]) for i in range(number_of_proj)]]

        # new head = shared_tail_id
        new_head_id = shared_tail_id
        new_edges = umls[(umls['head_id'] == new_head_id) & (umls['relation_id'] != 0)]

        if new_edges.empty:
            continue
        
        # Group tails by relations
        relation_dict = (
            new_edges.groupby('relation_id')['tail_id']
            .apply(list)
            .to_dict()
        )
        
        for rel, tails in relation_dict.items():
            queries.append(query+[rel])
            results.append(tails)

In [ ]:
with open(f'queries/{DATA}/queries-big.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

queries = loaded_dict['queries']
results = loaded_dict['results']

In [ ]:
qs.set_k(k_neighbors=50, k_results=25)

recalls = {
    "recall@1": [],
    "recall@5": [],
    "recall@10": [],
    "recall@25": [],
    "recall@50": [],
}

maps = {
    'map@1': [],
    'map@5': [],
    'map@10': [],
    'map@25': [],
    'map@50': [],
}

hits = {
    'hits@1': [],
    'hits@5': [],
    'hits@10': [],
    'hits@25': [],
    'hits@50': [],
}

mrr = []

for query, result in tqdm(zip(queries[:5000], results[:5000]), total=len(queries)):

    res = qs.execute_query(query, proj_mode="inter", agg_mode="union")
    result = np.array(list(result[-1]))

    if len(res) > 0:
        for k in [1, 5, 10, 25, 50]:
            recalls[f"recall@{k}"].append(recall_at_k(res, result, k))
            maps[f'map@{k}'].append(map_at_k(res, result, k))
            hits[f'hits@{k}'].append(custom_hits_at_k(res, result, k))
        mrr.append(custom_mrr(res, result))

print(f"Final results for {len(queries)} complex queries:")
print(f"Mrr: {np.mean(mrr):.4f}")
print("-----------------------")
print_metrics(hits, "Hits", [1, 5, 10, 25, 50])
print_metrics(recalls, "Recall", [1, 5, 10, 25, 50])
print_metrics(maps, "Map", [1, 5, 10, 25, 50])